In [ ]:
import pandas as pd
import ast

df = pd.read_csv("extended_dataset_2.csv")

df = df[
    df['min_storage'].notnull() &
    (df['min_storage'] <= 250) &
    (df['name'].str.lower().str.contains('playtest') == False)
].copy()

In [ ]:
df["parsed_og"] = df["og_string"].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) else {}
)

df["genres"] = df["parsed_og"].apply(
    lambda x: x.get("genres", []) if isinstance(x, dict) else []
)

print(df["genres"].head(10))

In [ ]:
from collections import Counter

genre_counter = Counter()

for genres in df["genres"]:
    if isinstance(genres, list):
        genre_counter.update(genres)

print(genre_counter.most_common(20))

In [ ]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

df["release_year"] = df["release_date"].dt.year

print(df["rec_storage"].describe())
print(df["min_storage"].describe())

In [ ]:
df["rec_storage_clean"] = df["rec_storage"].where(
    (df["rec_storage"] > 0) &
    (df["rec_storage"] < 250)
)

df["min_storage_clean"] = df["min_storage"].where(
    (df["min_storage"] > 0) &
    (df["min_storage"] < 250)
)

print(df["rec_storage_clean"].describe())
print(df["min_storage_clean"].describe())

In [ ]:
df_exploded = df.explode("genres")

selected_genres = [
    "Action",
    "Adventure",
    "RPG",
    "Casual",
    "Simulation",
    "Strategy",
    "Indie"
]

filtered = df_exploded[
    (df_exploded["genres"].isin(selected_genres)) &
    (df_exploded["release_year"] >= 2014) &
    (df_exploded["release_year"] <= 2024)
]

genre_trends = (
    filtered.groupby(["release_year", "genres"])["min_storage_clean"]
    .mean()
    .reset_index()
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,8))

for genre in selected_genres:
    
    subset = genre_trends[
        genre_trends["genres"] == genre
    ]
    
    plt.plot(
        subset["release_year"],
        subset["min_storage_clean"],
        label=genre,
        linewidth=2
    )

plt.xlabel("Release Year")
plt.ylabel("Average Minimum Storage (GB)")
plt.title("Genre-Specific Game Storage Growth Over Time")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
counts = (
    filtered.groupby(["release_year", "genres"])
    .size()
    .reset_index(name="count")
)

genre_trends = (
    filtered.groupby(["release_year", "genres"])["min_storage_clean"]
    .mean()
    .reset_index()
)

genre_trends = genre_trends.merge(
    counts,
    on=["release_year", "genres"]
)

genre_trends = genre_trends[
    genre_trends["count"] >= 20
]

plt.figure(figsize=(14,8))

for genre in selected_genres:

    subset = genre_trends[
        genre_trends["genres"] == genre
    ].sort_values("release_year")

    subset["smoothed"] = (
        subset["min_storage_clean"]
        .rolling(window=3, min_periods=1)
        .mean()
    )

    plt.plot(
        subset["release_year"],
        subset["smoothed"],
        label=genre,
        linewidth=2
    )

plt.xlabel("Release Year")
plt.ylabel("Average Minimum Storage (GB)")
plt.title("Genre-Specific Game Storage Growth Over Time")

plt.legend()
plt.grid(True)

plt.show()

# 1. Genre - storage boxplot
# 2. Genre - mean vs median trend
# 3. RAM vs storage

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import matplotlib.pyplot as plt

def parse_og_string(x):
    if pd.isna(x):
        return {}
    try:
        return ast.literal_eval(x)
    except:
        return {}

df["parsed_og"] = df["og_string"].apply(parse_og_string)

df["genres"] = df["parsed_og"].apply(
    lambda x: x.get("genres", []) if isinstance(x, dict) else []
)

# ============================================================
# 2. Clean release year
# ============================================================

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year

# Keep realistic release years only
df = df[
    (df["release_year"] >= 2005) &
    (df["release_year"] <= 2025)
].copy()

# ============================================================
# 3. Clean storage values
# ============================================================

df["min_storage_clean"] = df["min_storage"].where(
    (df["min_storage"] > 0) &
    (df["min_storage"] < 250)
)

df["rec_storage_clean"] = df["rec_storage"].where(
    (df["rec_storage"] > 0) &
    (df["rec_storage"] < 250)
)

# Main analysis uses min_storage_clean because rec_storage is sparse
print("Minimum storage summary:")
print(df["min_storage_clean"].describe())

print("\nRecommended storage summary:")
print(df["rec_storage_clean"].describe())

# ============================================================
# 4. Select genres and explode genre list
# ============================================================

selected_genres = [
    "Action",
    "Adventure",
    "Casual",
    "RPG",
    "Simulation",
    "Strategy",
    "Indie"
]

df_exploded = df.explode("genres")

filtered = df_exploded[
    (df_exploded["genres"].isin(selected_genres)) &
    (df_exploded["min_storage_clean"].notna()) &
    (df_exploded["release_year"] >= 2014) &
    (df_exploded["release_year"] <= 2024)
].copy()

# ============================================================
# 5. Analysis 1: Genre storage distribution boxplot
# ============================================================

boxplot_df = filtered[
    filtered["min_storage_clean"] < 100
].copy()

plt.figure(figsize=(12, 6))

boxplot_df.boxplot(
    column="min_storage_clean",
    by="genres",
    figsize=(12, 6)
)

plt.title("Storage Distribution by Genre")
plt.suptitle("")
plt.xlabel("Genre")
plt.ylabel("Minimum Storage (GB)")
plt.grid(True)

plt.show()

# ============================================================
# 6. Analysis 2: Mean vs Median storage trend by genre
# ============================================================

genre_stats = (
    filtered
    .groupby(["release_year", "genres"])["min_storage_clean"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

# Remove unstable year-genre groups with too few samples
genre_stats = genre_stats[
    genre_stats["count"] >= 20
].copy()

# Rolling smoothing
genre_stats = genre_stats.sort_values(["genres", "release_year"])

genre_stats["mean_smoothed"] = (
    genre_stats
    .groupby("genres")["mean"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

genre_stats["median_smoothed"] = (
    genre_stats
    .groupby("genres")["median"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

# Mean trend plot
plt.figure(figsize=(14, 8))

for genre in selected_genres:
    subset = genre_stats[genre_stats["genres"] == genre]
    plt.plot(
        subset["release_year"],
        subset["mean_smoothed"],
        label=genre,
        linewidth=2
    )

plt.xlabel("Release Year")
plt.ylabel("Average Minimum Storage (GB)")
plt.title("Mean Minimum Storage Growth by Genre")
plt.legend()
plt.grid(True)

plt.show()

# Median trend plot
plt.figure(figsize=(14, 8))

for genre in selected_genres:
    subset = genre_stats[genre_stats["genres"] == genre]
    plt.plot(
        subset["release_year"],
        subset["median_smoothed"],
        label=genre,
        linewidth=2
    )

plt.xlabel("Release Year")
plt.ylabel("Median Minimum Storage (GB)")
plt.title("Median Minimum Storage Growth by Genre")
plt.legend()
plt.grid(True)

plt.show()

# ============================================================
# 7. Analysis 3: Parse RAM values
# ============================================================

def parse_ram_to_gb(x):
    if pd.isna(x):
        return np.nan
    
    text = str(x).lower()
    
    # Remove commas
    text = text.replace(",", "")
    
    # Match GB
    gb_match = re.search(r"(\d+(\.\d+)?)\s*gb", text)
    if gb_match:
        return float(gb_match.group(1))
    
    # Match MB
    mb_match = re.search(r"(\d+(\.\d+)?)\s*mb", text)
    if mb_match:
        return float(mb_match.group(1)) / 1024
    
    return np.nan

df["min_ram_gb"] = df["mat_pc_memory_min"].apply(parse_ram_to_gb)
df["rec_ram_gb"] = df["mat_pc_memory_rec"].apply(parse_ram_to_gb)

# Clean unrealistic RAM values
df["min_ram_gb_clean"] = df["min_ram_gb"].where(
    (df["min_ram_gb"] > 0) &
    (df["min_ram_gb"] <= 256)
)

df["rec_ram_gb_clean"] = df["rec_ram_gb"].where(
    (df["rec_ram_gb"] > 0) &
    (df["rec_ram_gb"] <= 256)
)

print("\nMinimum RAM summary:")
print(df["min_ram_gb_clean"].describe())

print("\nRecommended RAM summary:")
print(df["rec_ram_gb_clean"].describe())

# ============================================================
# 8. RAM vs storage correlation
# ============================================================

#include a line of best fit in the scatter plot

ram_storage_df = df[
    df["min_storage_clean"].notna() &
    df["min_ram_gb_clean"].notna()
].copy()

print("\nRAM-storage correlation:")
print(
    ram_storage_df[["min_storage_clean", "min_ram_gb_clean"]]
    .corr()
)

plt.figure(figsize=(10, 6))

plt.scatter(
    ram_storage_df["min_ram_gb_clean"],
    ram_storage_df["min_storage_clean"],
    alpha=0.2,

)

plt.xlabel("Minimum RAM (GB)")
plt.ylabel("Minimum Storage (GB)")
plt.xlim(0, 32)
plt.ylim(0, 250)
plt.title("Relationship Between Minimum RAM and Storage Requirements")
plt.grid(True)

m, b = np.polyfit(ram_storage_df["min_ram_gb_clean"], ram_storage_df["min_storage_clean"], 1)

# 4. Add the line of best fit to the plot
plt.plot(ram_storage_df["min_ram_gb_clean"], m * ram_storage_df["min_ram_gb_clean"] + b, color="red", linestyle="--", label="trendline")
print(f"Line of best fit: storage = {m:.2f} * RAM + {b:.2f}")

plt.show()

# ============================================================
# 9. RAM trend over time
# ============================================================

ram_trend = (
    df[df["min_ram_gb_clean"].notna()]
    .groupby("release_year")["min_ram_gb_clean"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

ram_trend = ram_trend[
    ram_trend["count"] >= 20
].copy()

ram_trend["mean_smoothed"] = (
    ram_trend["mean"]
    .rolling(window=3, min_periods=1)
    .mean()
)

ram_trend["median_smoothed"] = (
    ram_trend["median"]
    .rolling(window=3, min_periods=1)
    .mean()
)

plt.figure(figsize=(12, 6))

plt.plot(
    ram_trend["release_year"],
    ram_trend["mean_smoothed"],
    label="Mean RAM",
    linewidth=2
)

plt.plot(
    ram_trend["release_year"],
    ram_trend["median_smoothed"],
    label="Median RAM",
    linewidth=2
)

plt.xlabel("Release Year")
plt.ylabel("Minimum RAM (GB)")
plt.title("Minimum RAM Requirements Over Time")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:

df_exploded["genres"].value_counts().head(20)

In [ ]:
#graph relative counts of genres over time
genre_counts = (
    filtered[filtered["genres"].isin(selected_genres)]
    .groupby(["release_year", "genres"])
    .size()
    .reset_index(name="count")
)
plt.figure(figsize=(14,8))
for genre in selected_genres:
    
    subset = genre_counts[
        genre_counts["genres"] == genre
    ]
    
    plt.plot(
        subset["release_year"],
        subset["count"],
        label=genre,
        linewidth=2
    )
plt.xlabel("Release Year")
plt.ylabel("Count")
plt.title("Genre Trends Over Time")
plt.legend()
plt.show()

In [ ]:
#plot indie games as a fraction of all games over time
total_counts = (
    filtered.groupby("release_year")
    .size()
    .reset_index(name="total_count")
)
indie_counts = (
    filtered[filtered["genres"] == "Indie"]
    .groupby("release_year")
    .size()
    .reset_index(name="indie_count")
)   
indie_trend = indie_counts.merge(
    total_counts,
    on="release_year"
)
indie_trend["indie_fraction"] = indie_trend["indie_count"] / indie_trend["total_count"]
plt.figure(figsize=(12, 6))
plt.plot(
    indie_trend["release_year"],
    indie_trend["indie_fraction"],
    label="Indie Fraction",
    linewidth=2
)